# KMeans with Log Transform

This notebook applies **log transformation** to skewed numerical features before running **KMeans clustering**.

| Step | Description |
|---|---|
| **Log Transform** | `np.log1p` compresses large value ranges and reduces skew, making distance-based algorithms more effective |
| **StandardScaler** | Normalises features to zero mean and unit variance after log transform |
| **KMeans (k=3)** | Groups products into 3 clusters based on cost and discount behaviour |

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

## 2. Load Dataset

In [ ]:
df = pd.read_csv("Train.csv")

print("Dataset Shape:", df.shape)
df.head()

## 3. Data Quality Check

In [ ]:
print("Missing Values:\n")
print(df.isnull().sum())

## 4. Select & Clean Clustering Features

In [ ]:
cluster_df = df[['Cost_of_the_Product', 'Discount_offered']]
cluster_df.head()

In [ ]:
cluster_df = cluster_df.dropna()

print("Shape after removing null values:", cluster_df.shape)

## 5. Log Transformation

`np.log1p(x)` = `log(1 + x)` — safe for zero values, compresses right-skewed distributions.

In [ ]:
cluster_df = cluster_df.copy()
cluster_df['Cost_log'] = np.log1p(cluster_df['Cost_of_the_Product'])
cluster_df['Discount_log'] = np.log1p(cluster_df['Discount_offered'])

cluster_df[['Cost_log', 'Discount_log']].head()

### 5.1 Before vs After Log Transform — Distribution

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
sns.histplot(cluster_df['Cost_of_the_Product'], kde=True)
plt.title("Before Log Transform")

plt.subplot(1, 2, 2)
sns.histplot(cluster_df['Cost_log'], kde=True)
plt.title("After Log Transform")

plt.tight_layout()
plt.show()

## 6. Standardise & Fit KMeans

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(cluster_df[['Cost_log', 'Discount_log']])

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X)
cluster_df['Cluster'] = clusters

cluster_df.head()

## 7. Cluster Profiles

In [ ]:
for i in range(3):
    print("\nCluster", i)
    print(cluster_df[cluster_df['Cluster'] == i].describe())

## 8. Visualise Cluster Distributions

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(x='Cluster', y='Cost_log', data=cluster_df)
plt.title("Cost Distribution per Cluster (Log Scale)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(x='Cluster', y='Discount_log', data=cluster_df)
plt.title("Discount Distribution per Cluster (Log Scale)")
plt.tight_layout()
plt.show()

## 9. Cluster Centers (Standardised Space)

In [ ]:
print("Cluster Centers:\n")
print(kmeans.cluster_centers_)